# Stock Image Scraper 📸

In [ ]:
# STEP 1:- Importing the required libraries
import time
import requests
import pandas as pd
from tqdm import tqdm
import chromedriver_binary
from bs4 import BeautifulSoup
from selenium import webdriver
from openpyxl import Workbook
import re
import os
import urllib.request
from urllib.parse import urljoin

In [6]:
# STEP :-2:- Setting up the Selenium WebDriver
driver = webdriver.Chrome()
driver.get("https://stock-pictures.netlify.app/")
time.sleep(5)  # Wait for the page to load

The chromedriver version (147.0.7727.57) detected in PATH at c:\Users\Nbinary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (148.0.7778.181); currently, chromedriver 148.0.7778.178 is recommended for chrome 148.*, so it is advised to delete the driver in PATH and retry


In [7]:
# STEP 3:- Scrapping the url
soup = BeautifulSoup(driver.page_source, "html.parser")
image_elements = soup.select('img.source-img[src$=".jpg"]')
print(f"Found {len(image_elements)} images on the page.")

Found 48 images on the page.


In [8]:
# STEP 3:- Scraping the image details
# image url, name tags, likes and comments on each image
image_data = []
for sp in soup.find_all('div', class_='container'):
    img = sp.find('img')
    
    # 1. Safely check if img exists AND has a src attribute before proceeding
    if img and img.get('src') and 'gif' not in img.get('src'):
        link = img.get('src')
        
        # 2. Safely extract tags (Default to empty string if missing)
        tags_div = sp.find('div', class_='tags')
        tags_text = ""
        if tags_div:
            # Assuming the first 7 chars were something like "Tags: "
            # A safer way is replacing the specific word, or just matching words
            raw_tags = tags_div.text[7:].strip().split(' ')
            tags_text = ' '.join(list(set(raw_tags)))
        
        # 3. Safely extract likes and comments (Default to 0 if missing)
        likes, comments = 0, 0
        likes_comments_div = sp.find('div', class_='likes-comments')
        
        if likes_comments_div:
            spans = likes_comments_div.find_all('span')
            
            # re.search(r'\d+', text) finds the first sequence of numbers in a string
            if len(spans) > 0:
                likes_match = re.search(r'\d+', spans[0].text)
                likes = int(likes_match.group()) if likes_match else 0
                
            if len(spans) > 1:
                comments_match = re.search(r'\d+', spans[1].text)
                comments = int(comments_match.group()) if comments_match else 0
        
        image_data.append([link, tags_text, likes, comments])

In [9]:
# STEP 4: Make in to a dataframe
df = pd.DataFrame(image_data, columns=['Image URL', 'Tags', 'Likes', 'Comments'])

In [10]:
df

,Image URL,Tags,Likes,Comments
0,https://cdn.pixabay.com/photo/2022/03/06/05/30...,"Sky Atmosphere, Blue Sky, Clouds,",196,55
1,https://cdn.pixabay.com/photo/2022/04/07/11/45...,"Ornithology, Bird, Hummingbird",76,20
2,https://cdn.pixabay.com/photo/2022/02/28/15/28...,"Sea, Rainbow, Rainfall, Subtropical",282,106
3,https://cdn.pixabay.com/photo/2022/04/04/02/52...,"Japan, Road, Sakura Cherry Blossoms,",42,11
4,https://cdn.pixabay.com/photo/2022/04/09/18/06...,"Flower, Marguerite, Cape Plant",39,15
5,https://cdn.pixabay.com/photo/2021/11/13/23/06...,Rest Under Relaxing The Tree,522,108
6,https://cdn.pixabay.com/photo/2022/04/07/02/56...,"Grass Rabbit, Wild Cottontail",51,10
7,https://cdn.pixabay.com/photo/2022/03/19/21/11...,"Flowers, Crocus, Plant Spring,",135,44
8,https://cdn.pixabay.com/photo/2022/04/09/17/30...,"Café, Vacation, Drink, Coffee, Table",26,5
9,https://cdn.pixabay.com/photo/2022/03/25/19/24...,"Waterfall, Light Epic, Nature, Fall,",96,21


In [11]:
# STEP 5: Close the Selenium WebDriver
driver.quit()

In [12]:
# STEP 6: Save the data to an Excel file
df.to_excel("stock_images_data.xlsx", index=False)

In [14]:
# STEP 7: Download the images to a local folder
# Create a folder to save the images
os.makedirs("downloaded_images", exist_ok=True)
# Use a browser-like User-Agent to avoid being blocked by the server
download_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}
# Download each image
for index, row in tqdm(df.iterrows(), total=len(df)):
    image_url = row['Image URL']
    if image_url.startswith('/'):
        image_url = urljoin("https://stock-pictures.netlify.app/", image_url)
    image_name = f"image_{index + 1}.jpg"
    save_path = os.path.join("downloaded_images", image_name)
    try:
        response = requests.get(image_url, headers=download_headers, timeout=30, stream=True)
        response.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=10240):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded: {image_name}")
    except Exception as e:
        print(f"Failed to download {image_name}: {image_url} : {e}")

  0%|          | 0/48 [00:00<?, ?it/s]

  2%|▏         | 1/48 [00:01<00:48,  1.04s/it]

Downloaded: image_1.jpg


  4%|▍         | 2/48 [00:02<00:49,  1.08s/it]

Downloaded: image_2.jpg


...

Downloaded: image_46.jpg


 98%|█████████▊| 47/48 [00:46<00:00,  1.37it/s]

Downloaded: image_47.jpg


100%|██████████| 48/48 [00:47<00:00,  1.02it/s]

Downloaded: image_48.jpg
